# Veritas: fix the "obviously true claims marked Contradicted" bug, then launch the demo
One notebook, three parts. **Runtime -> Run all** (T4 GPU, about 35-40 minutes total). Approve the Drive pop-up once at the start.

| Part | What it does | Time | Saves to Drive |
|---|---|---|---|
| 1. Verdict model v3 | Continues training v2 with irrelevant-abstract NEI examples; checks it did not regress | ~20 min | `verdict_model_v3/`, `logits_v3.npz` |
| 2. Reranker + aggregator | Re-fits the aggregator on v3, picks the abstain gate | ~10 min | `retrieval/aggregator.joblib` (old one kept as `aggregator_prev.joblib`), `retrieval_config.json` |
| 3. Demo | Gradio app; prints a public link | ~5 min | nothing |

If v3 drops more than 5 macro-F1 points vs v2 on HealthVer TEST or SciFact val, the notebook automatically keeps v2 for Parts 2 and 3 (v3 is still saved) and says so.

**Paste back:** Part 1 checks 1-3, Part 2 gate table + free-text claims, a screenshot of the demo.

**Runs on Colab or Kaggle.** Kaggle: add your `veritas` dataset as an input, turn Internet on, pick a GPU. Results are written to `/kaggle/working/veritas`.

**If the session disconnects mid-way:** each part saves its result to the veritas folder. Reconnect, run the install cell and the location cell, then run from the header of the part that did not finish. Part 2 and Part 3 each have their own setup cell, so they do not need Part 1's variables.

In [ ]:
%pip install -q "datasets==3.6.0" "transformers>=4.46,<5" accelerate sentencepiece protobuf scikit-learn sentence-transformers rank_bm25 pandas pyarrow joblib youtube-transcript-api "gradio>=5,<6"

In [ ]:
import os, shutil
# Where the veritas folder lives. Colab: Google Drive. Kaggle: copied from your uploaded dataset into /kaggle/working (inputs are read-only).
# Colab is checked FIRST: Colab also has an (empty) /kaggle/input folder.
try:
    from google.colab import drive
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
if ON_COLAB:
    drive.mount("/content/drive"); DRIVE_ROOT = "/content/drive/MyDrive/veritas"
elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE") and os.path.isdir("/kaggle/input"):   # only real Kaggle sessions set this variable
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # use one GPU so training matches the Colab runs (set before torch loads)
    DRIVE_ROOT = "/kaggle/working/veritas"
    if not os.path.isdir(f"{DRIVE_ROOT}/verdict_model_v2"):
        src = None
        for dp, dn, fn in os.walk("/kaggle/input"):
            if "verdict_model_v2" in dn: src = dp; break
            if dp.count(os.sep) > 6: dn[:] = []
        assert src, "No folder containing verdict_model_v2 found under /kaggle/input. Add your veritas dataset (Add Input) and rerun."
        print("copying", src, "->", DRIVE_ROOT)
        shutil.copytree(src, DRIVE_ROOT, dirs_exist_ok=True, ignore=shutil.ignore_patterns("ckpt*"))
    import urllib.request
    try: urllib.request.urlopen("https://huggingface.co", timeout=8); print("internet: on")
    except Exception: raise SystemExit("Internet is off. Kaggle: right sidebar -> Session options -> Internet -> On (needs a phone-verified account), then rerun.")
else:
    DRIVE_ROOT = "veritas_local"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("veritas folder:", DRIVE_ROOT, "|", sorted(os.listdir(DRIVE_ROOT)))

# Part 1 of 3 — Step 2c: Verdict model v3 — teach it that irrelevant evidence means NEI
**The bug.** In the demo, "if you don't drink water, you will die" came back *Contradicted (70%)*. The best abstract it found was an alcohol-burden paper, and the verdict model called it "refutes" at 91%. The model was only ever trained on abstracts that are *about* the claim, so it forces a verdict onto any text.

**The fix (this notebook).** Continue training v2 with extra **NEI** examples where the evidence is *not* evidence:
- **hard negatives**: for each SciFact/HealthVer train claim, an abstract that search ranks 10th–60th (topically near, but not one of the claim's labeled abstracts),
- **random negatives**: a random abstract from the pool,
- **PUBHEALTH claims paired with random abstracts**.

**Checks (all held out from training)**
1. **Regression**: v2 vs v3 macro-F1 on the same five evaluation sets as step 2. v3 must not collapse.
2. **Irrelevant-pair test**: validation claims paired with near-miss abstracts. v3 should say NEI far more often, and be "confidently wrong" far less often.
3. **Probe claims**: obvious off-pool claims like the water one, before and after.

Saves `verdict_model_v3/`. v2 is left alone. **Runtime → T4 GPU**, approve the Drive pop-up. About 20 minutes.

In [ ]:
import os, random, numpy as np, pandas as pd, torch, torch.nn.functional as F
from datasets import Dataset
from sklearn.metrics import f1_score
from sentence_transformers import SentenceTransformer
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments)

SMOKE = False                 # True = tiny local check of the code
EPOCHS, LR, BATCH, MAX_LEN, SEED = 2, 1e-5, 16, 384, 42
SCIFACT_REPEAT = 3
N_HARD, N_RAND, PH_RAND = 1, 1, 1500   # negatives per SciFact/HealthVer claim; PUBHEALTH claims paired with a random abstract
HARD_LO, HARD_HI = 10, 60              # hard negatives come from search ranks 10-60 (ranks 0-9 may be real evidence)
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
DEV = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if not SMOKE: assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
DRIVE = DRIVE_ROOT   # set by the location cell near the top
INIT, OUT = f"{DRIVE}/verdict_model_v2", f"{DRIVE}/verdict_model_v3"
CKPT = "/content/ckpt_v3" if os.path.isdir("/content") else "/tmp/ckpt_v3"   # local disk: training is short, saves Drive space
print("device:", DEV, "| start from:", INIT)

## 1. Data and the evidence pool

In [ ]:
df = pd.read_parquet(f"{DRIVE}/unified_v2.parquet" if os.path.exists(f"{DRIVE}/unified_v2.parquet") else "unified_v2.parquet")
get = lambda src, sp: df[(df.source == src) & (df.split == sp)].reset_index(drop=True)
hv_train, hv_val, hv_test = get("healthver", "train"), get("healthver", "validation"), get("healthver", "test")
hv_train = hv_train[~hv_train.claim.isin(set(hv_val.claim) | set(hv_test.claim))]
sf_train, sf_val = get("scifact", "train"), get("scifact", "validation")
ph_train, ph_val, ph_test = get("pubhealth", "train"), get("pubhealth", "validation"), get("pubhealth", "test")

corpus = pd.read_parquet(f"{DRIVE}/retrieval/corpus.parquet"); D = np.load(f"{DRIVE}/retrieval/corpus_emb.npy")
assert len(corpus) == len(D)
st = SentenceTransformer("BAAI/bge-small-en-v1.5", device=DEV); st.max_seq_length = 256
PREFIX = "Represent this sentence for searching relevant passages: "
ptexts = corpus.text.str.strip().tolist()
print({k: len(v) for k, v in dict(sf_train=sf_train, hv_train=hv_train, ph_train=ph_train, pool=corpus).items()})

## 2. Mine the negatives
For each claim, search the pool (dense). Anything the dataset already pairs with that claim (any label) is excluded, so a true evidence abstract is never relabeled NEI.

In [ ]:
def neighbours(claims):
    Q = st.encode([PREFIX + c for c in claims], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    return np.argsort(-(Q @ D.T), axis=1)[:, :HARD_HI]

def mine(rows, n_hard, n_rand, rng, lo=HARD_LO, hi=HARD_HI):
    gold = rows.groupby("claim").evidence.apply(lambda s: {x.strip() for x in s}).to_dict()
    claims = list(gold); nb = neighbours(claims); out = []
    for c, order in zip(claims, nb):
        cand = [j for j in order[lo:hi] if ptexts[j] not in gold[c]]
        for j in (rng.choice(cand, size=min(n_hard, len(cand)), replace=False) if cand else []):
            out.append(dict(claim=c, evidence=corpus.text.iat[j], label=2, source="neg_hard"))
        for _ in range(n_rand):
            j = int(rng.randint(len(corpus)))
            while ptexts[j] in gold[c]: j = int(rng.randint(len(corpus)))
            out.append(dict(claim=c, evidence=corpus.text.iat[j], label=2, source="neg_rand"))
    return pd.DataFrame(out)

rng = np.random.RandomState(SEED)
hard = mine(pd.concat([sf_train, hv_train]), N_HARD, N_RAND, rng)
ph_neg = pd.DataFrame([dict(claim=c, evidence=corpus.text.iat[int(rng.randint(len(corpus)))], label=2, source="neg_ph")
                       for c in ph_train.claim.sample(min(PH_RAND, len(ph_train)), random_state=SEED)])
neg = pd.concat([hard, ph_neg]).reset_index(drop=True)
print(neg.source.value_counts().to_dict())
r = neg[neg.source == "neg_hard"].sample(2, random_state=1)
for _, x in r.iterrows(): print("\nCLAIM:", x.claim, "\n  NEAR-MISS ABSTRACT:", x.evidence[:200], "...")

## 3. Train
Starts from v2's weights, so this is a short continuation (2 epochs, small learning rate), not a retrain from scratch. The class weights are recomputed because NEI is now much more common.

In [ ]:
base = pd.concat([ph_train, pd.concat([sf_train] * SCIFACT_REPEAT), hv_train])[["claim", "evidence", "label"]]
train_df = pd.concat([base, neg[["claim", "evidence", "label"]]]).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df = pd.concat([ph_val, sf_val, hv_val])[["claim", "evidence", "label"]].reset_index(drop=True)
if SMOKE:
    train_df = train_df.sample(300, random_state=0).reset_index(drop=True); MAX_LEN = 128
    ph_val, sf_val, hv_val, ph_test, hv_test = [x.head(40) for x in (ph_val, sf_val, hv_val, ph_test, hv_test)]
    val_df = val_df.sample(120, random_state=0).reset_index(drop=True)
cnt = train_df.label.value_counts().reindex([0, 1, 2]).values
cw = torch.tensor((len(train_df) / (3 * cnt)) ** 0.5, dtype=torch.float)
print("train rows:", len(train_df), "| class share:", dict(zip(LABELS, (cnt / cnt.sum()).round(3))), "| weights:", dict(zip(LABELS, cw.numpy().round(2))))

tok = AutoTokenizer.from_pretrained(INIT)
def make_ds(d):
    ds = Dataset.from_pandas(d[["claim", "evidence", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return ds.map(lambda b: tok(b["claim"], b["evidence"], truncation="only_second", max_length=MAX_LEN), batched=True, remove_columns=["claim", "evidence"])
train_ds, val_ds = make_ds(train_df), make_ds(val_df)

class WeightedTrainer(Trainer):
    def __init__(self, *a, class_weights=None, **k): super().__init__(*a, **k); self.cw = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs)
        loss = F.cross_entropy(out.logits.float(), labels, weight=self.cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss
def compute_metrics(p):
    pr = p.predictions.argmax(-1); return {"macro_f1": f1_score(p.label_ids, pr, average="macro"), "acc": float((pr == p.label_ids).mean())}

model = AutoModelForSequenceClassification.from_pretrained(INIT)
args = TrainingArguments(output_dir=CKPT, num_train_epochs=1 if SMOKE else EPOCHS, learning_rate=LR, per_device_train_batch_size=BATCH,
                         per_device_eval_batch_size=BATCH * 2, warmup_ratio=0.1, weight_decay=0.01, fp16=torch.cuda.is_available(),
                         eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
                         metric_for_best_model="macro_f1", logging_steps=100, report_to="none", seed=SEED)
trainer = WeightedTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                          data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics, class_weights=cw)
trainer.train()

## 4. Check 1: did v3 hold up on the original task?
Same five evaluation sets as step 2, scored by both models in the same way. A drop of a couple of points is acceptable. A collapse is not.

In [ ]:
def predict(m, d):
    t = Trainer(model=m, data_collator=DataCollatorWithPadding(tok),
                args=TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=32, fp16=torch.cuda.is_available(), report_to="none"))
    o = t.predict(make_ds(d)); return o.predictions, o.label_ids
v2 = AutoModelForSequenceClassification.from_pretrained(INIT)
sets = {"PUBHEALTH val": ph_val, "SciFact val": sf_val, "HealthVer val": hv_val, "PUBHEALTH test": ph_test, "HealthVer TEST": hv_test}
rows, res3 = [], {}
for name, d in sets.items():
    for tag, m in [("v2", v2), ("v3", trainer.model)]:
        lg, y = predict(m, d); p = lg.argmax(-1)
        if tag == "v3": res3[name] = (lg, y)
        rows.append(dict(set=name, model=tag, macro_F1=round(f1_score(y, p, average="macro"), 3), acc=round((p == y).mean(), 3),
                         NEI_recall=round(((p == 2) & (y == 2)).sum() / max((y == 2).sum(), 1), 3)))
print(pd.DataFrame(rows).to_string(index=False))
CHECK1 = pd.DataFrame(rows)

## 5. Check 2: irrelevant evidence
Validation claims (never used for mining) paired with a near-miss abstract (search rank 10-60, not labeled evidence for that claim). The right answer is **NEI**. *Confidently wrong* = SUPPORTS or REFUTES at 80%+ probability.

In [ ]:
neg_eval = mine(pd.concat([sf_val, hv_val]), 1, 0, np.random.RandomState(7))
if SMOKE: neg_eval = neg_eval.head(60)
def probs_for(m, d):
    lg, _ = predict(m, d.assign(label=2)); z = lg - lg.max(-1, keepdims=True); return np.exp(z) / np.exp(z).sum(-1, keepdims=True)
rows = []
for tag, m in [("v2", v2), ("v3", trainer.model)]:
    P = probs_for(m, neg_eval); wrong = (P[:, :2].max(1) >= 0.8)
    rows.append(dict(model=tag, pairs=len(P), says_NEI=f"{(P.argmax(1) == 2).mean():.1%}", confidently_wrong=f"{wrong.mean():.1%}"))
print(pd.DataFrame(rows).to_string(index=False))

## 6. Check 3: probe claims
Obvious claims the pool cannot settle, plus a few that it might. For each: the top 5 abstracts by search, then the **mean P(NEI)** and the number of abstracts judged SUPPORTS/REFUTES at 80%+. Before and after.

In [ ]:
PROBES = ["If you don't drink water, you will die.", "If you do not eat food, you will die.", "Breathing oxygen is necessary for human survival.",
          "Sleep is important for good health.", "Drinking lemon water cures cancer.", "Magnesium supplements cure insomnia.",
          "Eating carrots gives you night vision.", "Regular exercise improves heart health.", "Vitamin D supplementation reduces the risk of respiratory infections."]
nbrs = neighbours(PROBES); rows = []
for c, order in zip(PROBES, nbrs):
    d = pd.DataFrame(dict(claim=c, evidence=[corpus.text.iat[j] for j in order[:5]], label=2))
    r = dict(claim=c[:62])
    for tag, m in [("v2", v2), ("v3", trainer.model)]:
        P = probs_for(m, d); r[f"{tag} meanNEI"] = round(float(P[:, 2].mean()), 2); r[f"{tag} #confident"] = int((P[:, :2].max(1) >= 0.8).sum())
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

## 7. Save

In [ ]:
os.makedirs(OUT, exist_ok=True)
trainer.save_model(OUT); tok.save_pretrained(OUT)
np.savez(f"{DRIVE}/logits_v3.npz", **{f"{k}_logits": v[0] for k, v in res3.items()}, **{f"{k}_labels": v[1] for k, v in res3.items()})
print(sorted(os.listdir(OUT)))

In [ ]:
# Automatic safety check: only hand v3 to Part 2 if it did not regress on the original task.
c1 = CHECK1.pivot(index="set", columns="model", values="macro_F1")
drops = (c1["v2"] - c1["v3"]).loc[["HealthVer TEST", "SciFact val"]]
CHOICE = "verdict_model_v3" if bool((drops <= 0.05).all()) else "verdict_model_v2"
open(f"{DRIVE}/verdict_choice.txt", "w").write(CHOICE)
print("macro-F1 drop v2 -> v3 (negative = v3 better):", drops.round(3).to_dict())
print("=> Part 2 will use:", CHOICE, "" if CHOICE == "verdict_model_v3" else "(v3 regressed by more than 5 points, so v2 is kept; v3 is still saved on Drive)")

## What to check
1. **Check 1**: v3 macro-F1 within about 3 points of v2 on HealthVer TEST and SciFact val. `NEI_recall` will rise; that is intended.
2. **Check 2**: `says_NEI` should jump (v2 was low) and `confidently_wrong` should fall a lot.
3. **Check 3**: the water, food and oxygen probes should now have a high mean P(NEI) and zero or few confident abstracts. The vitamin D and exercise probes should *not* collapse to all-NEI, because those have real evidence.
4. If v3 hurts HealthVer TEST by more than about 5 points, say so: the fix is fewer negatives (lower `N_HARD`, `N_RAND`, `PH_RAND`), not abandoning the idea.
5. **Next:** rerun step 3b pointed at v3 (I'm updating it so it reads the model name from one setting and picks the abstain threshold properly).

In [ ]:
# Free the GPU and drop Part 1's variables before part 2
assert os.path.isdir(f"{DRIVE}/verdict_model_v3"), "Part 1 did not save verdict_model_v3; fix it before running part 2"
print("Part 1 saved to Drive.")
import gc
P1 = ("BATCH CKPT D DEV DRIVE EPOCHS HARD_HI HARD_LO INIT LABELS LR MAX_LEN N_HARD N_RAND OUT P PH_RAND PREFIX PROBES Q SCIFACT_REPEAT SEED SMOKE "
      "args base cand claims cnt corpus cw d df ds gold hard hv_test hv_train hv_val j labels lg loss model neg neg_eval nbrs o out p ph_neg ph_test "
      "ph_train ph_val pr ptexts r res3 rng rows sets sf_train sf_val st t tag tok train_df train_ds trainer v2 val_df val_ds wrong y z get").split()
for n in P1: globals().pop(n, None)   # Part 2 defines everything it needs itself
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
print("GPU memory in use (GB):", round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else "n/a (CPU)")


# Part 2 of 3 — Step 3b: Reranker + learned aggregator
Step 3 found two problems:
1. **Retrieval misses the right abstract** on ~29% of HealthVer claims (recall@5 = 0.71).
2. **False SUPPORTS**: claims with no evidence get a SUPPORTS verdict, because "max P(SUP) over 5 passages" gives every irrelevant passage another chance to fire.

**Fixes in this notebook**
1. Hybrid search pulls the top **30**, a **cross-encoder reranker** re-scores them, the top 5 go to the verdict model.
2. Replace the max rule with a small **logistic regression** over per-claim features (max/mean/top-1 verdict probabilities + reranker scores). It is trained on the two **validation** sets only, then scored on HealthVer test **once**.

Sections 1-4 below are the same as step 3. **This version uses verdict model v3** (set by `VERDICT_DIR` in the config cell). **Runtime → T4 GPU**, approve the Drive pop-up. About 10 minutes.

In [ ]:
import os, re, json, joblib, collections, numpy as np, pandas as pd, torch
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DRIVE = DRIVE_ROOT   # set by the location cell near the top
OUT = f"{DRIVE}/retrieval"; os.makedirs(OUT, exist_ok=True)
VERDICT_DIR = open(f"{DRIVE}/verdict_choice.txt").read().strip() if os.path.exists(f"{DRIVE}/verdict_choice.txt") else "verdict_model_v3"   # which verdict model to use (v3 = trained with irrelevant-evidence negatives)
MODEL_DIR = f"{DRIVE}/{VERDICT_DIR}"
EMB_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
TOPK = 5                    # passages handed to the verdict model
NCAND = 30                  # hybrid candidates the reranker re-scores
RERANK_MODEL = "BAAI/bge-reranker-base"
KS = (1, 5, 10)             # recall@k reported
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
print("device:", DEV, "| model dir exists:", os.path.isdir(MODEL_DIR))

## 1. The abstract pool
Each document has a `title` and an `abstract`.
- **Searching** uses `title + abstract` (more words to match).
- **The verdict model** is shown the `abstract` only, because that is what it was trained on.

Both datasets come straight from their full corpora, not from the training table (which only held abstracts attached to a claim).

In [ ]:
sf_corpus = load_dataset("allenai/scifact", "corpus", trust_remote_code=True)["train"]
hv_all = load_dataset("dwadden/healthver_entailment", trust_remote_code=True)

docs = {}
for d in sf_corpus:
    docs[f"sf_{d['doc_id']}"] = dict(source="scifact", title=d["title"] or "", text=" ".join(d["abstract"]).strip())
for sp in hv_all:
    for ex in hv_all[sp]:
        docs.setdefault(f"hv_{ex['abstract_id']}", dict(source="healthver", title=ex["title"] or "",
                                                         text=" ".join(ex["abstract"]).strip()))
corpus = pd.DataFrame([dict(uid=u, **v) for u, v in docs.items()])
corpus["search_text"] = corpus["title"] + ". " + corpus["text"]
print(corpus.source.value_counts().to_dict(), "| total:", len(corpus))
print("median abstract words:", int(corpus.text.str.split().str.len().median()))

## 2. Evaluation claims
Three sets, all *claims the verdict model never trained on*:
SciFact validation, HealthVer validation, HealthVer test.

- **Gold documents**: the abstracts that actually contain evidence for the claim.
- **Gold label**: SUPPORTS / REFUTES from that evidence, or NEI if the claim has no evidence anywhere.

Claims with no gold document can't be scored for retrieval, but they **are** included in the end-to-end verdict score (the model should say NEI).

In [ ]:
def build_claims(rows):
    # rows: list of (claim_id, claim_text, doc_uid_or_None, label_name_or_None)
    by = collections.OrderedDict()
    for cid, claim, uid, lab in rows:
        e = by.setdefault(cid, dict(claim=claim.strip(), gold=set(), labs=[]))
        if uid is not None and lab in ("SUPPORTS", "REFUTES"):
            e["gold"].add(uid); e["labs"].append(lab)
    out = []
    for e in by.values():
        lab = collections.Counter(e["labs"]).most_common(1)[0][0] if e["labs"] else "NEI"
        out.append(dict(claim=e["claim"], gold=e["gold"], label=LABELS.index(lab)))
    return out

sf = load_dataset("allenai/scifact", "claims", trust_remote_code=True)["validation"]
m = {"SUPPORT": "SUPPORTS", "CONTRADICT": "REFUTES"}
sf_rows = [(ex["id"], ex["claim"], f"sf_{ex['evidence_doc_id']}" if ex["evidence_label"] else None, m.get(ex["evidence_label"])) for ex in sf]

hm = {"SUPPORT": "SUPPORTS", "CONTRADICT": "REFUTES", "NEI": "NEI"}
def hv_rows(split):
    return [(ex["claim_id"], ex["claim"], f"hv_{ex['abstract_id']}", hm.get(ex["verdict"])) for ex in hv_all[split]]

EVAL = {"SciFact val": build_claims(sf_rows),
        "HealthVer val": build_claims(hv_rows("validation")),
        "HealthVer test": build_claims(hv_rows("test"))}
for k, v in EVAL.items():
    n_gold = sum(bool(c["gold"]) for c in v)
    print(f"{k:15s} claims: {len(v):4d} | with gold evidence: {n_gold:4d} | NEI-only: {len(v) - n_gold:4d}")

## 3. Two searchers
- **BM25**: scores documents by shared rare words. Great for exact terms ("remdesivir"), blind to paraphrase.
- **Dense**: turns claim and abstract into vectors; nearby vectors = similar meaning. Handles paraphrase, can miss exact terms.
- **Hybrid (RRF)**: each searcher ranks the pool; documents ranked high by both win. Fusion score = Σ 1 / (60 + rank).

In [ ]:
tok_re = re.compile(r"[a-z0-9]+")
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())
def toks(t): return [w for w in tok_re.findall(t.lower()) if w not in STOP]

bm25 = BM25Okapi([toks(t) for t in corpus.search_text])

st = SentenceTransformer(EMB_MODEL, device=DEV); st.max_seq_length = 256
emb_path = f"{OUT}/corpus_emb.npy"
if os.path.exists(emb_path) and len(np.load(emb_path, mmap_mode="r")) == len(corpus):
    D = np.load(emb_path)
else:
    D = st.encode(corpus.search_text.tolist(), batch_size=64, normalize_embeddings=True,
                  show_progress_bar=True, convert_to_numpy=True).astype("float32")
print("doc embeddings:", D.shape)

def rank_bm25(queries, n=100):
    return [np.argsort(-bm25.get_scores(toks(q)))[:n] for q in queries]

def rank_dense(queries, n=100):
    Q = st.encode([QUERY_PREFIX + q for q in queries], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    S = Q @ D.T
    return [np.argsort(-s)[:n] for s in S]

def fuse(r1, r2, n=100, k=60):
    out = []
    for a, b in zip(r1, r2):
        sc = collections.defaultdict(float)
        for r in (a, b):
            for rank, i in enumerate(r): sc[i] += 1.0 / (k + rank + 1)
        out.append(np.array(sorted(sc, key=sc.get, reverse=True)[:n]))
    return out

uid_of = corpus.uid.values

## 4. Recall@k: is the right abstract in the top k?

In [ ]:
RANKS = {}   # (set name) -> dict(method -> list of rankings)
rows = []
for name, claims in EVAL.items():
    qs = [c["claim"] for c in claims]
    rb, rd = rank_bm25(qs), rank_dense(qs)
    RANKS[name] = {"BM25": rb, "Dense": rd, "Hybrid": fuse(rb, rd)}
    has_gold = [i for i, c in enumerate(claims) if c["gold"]]
    for meth, R in RANKS[name].items():
        rec = {k: np.mean([bool(set(uid_of[R[i][:k]]) & claims[i]["gold"]) for i in has_gold]) for k in KS}
        rows.append(dict(set=name, method=meth, **{f"recall@{k}": round(v, 3) for k, v in rec.items()}))
print(pd.DataFrame(rows).to_string(index=False))

## 5. Rerank the top 30
The searchers embed claim and abstract **separately** and compare vectors. A cross-encoder reads the claim and the abstract **together**, which is slower but far better at "does this abstract actually address this claim". So: cheap search for 30 candidates, expensive reranker to pick the best 5.

In [ ]:
from sentence_transformers import CrossEncoder
rr = CrossEncoder(RERANK_MODEL, max_length=384, device=DEV)
if DEV == "cuda": rr.model.half()

def rerank(claims, rankings, ncand=NCAND, bs=64):
    pairs = [(c["claim"], corpus.search_text.iat[di]) for c, r in zip(claims, rankings) for di in r[:ncand]]
    sc = np.asarray(rr.predict(pairs, batch_size=bs, show_progress_bar=False), dtype="float64").reshape(len(claims), -1)
    if sc.min() < 0 or sc.max() > 1: sc = 1 / (1 + np.exp(-sc))     # keep scores in (0, 1)
    new_r, new_s = [], []
    for r, s in zip(rankings, sc):
        o = np.argsort(-s); new_r.append(r[:ncand][o]); new_s.append(s[o])
    return new_r, np.array(new_s)

RERANKED, RSCORE = {}, {}
for name, claims in EVAL.items():
    RERANKED[name], RSCORE[name] = rerank(claims, RANKS[name]["Hybrid"])

rows = []
for name, claims in EVAL.items():
    has_gold = [i for i, c in enumerate(claims) if c["gold"]]
    for meth, R, ks in [("Hybrid", RANKS[name]["Hybrid"], (1, 5, 10, NCAND)), ("Hybrid+rerank", RERANKED[name], (1, 5, 10))]:
        rec = {k: np.mean([bool(set(uid_of[R[i][:k]]) & claims[i]["gold"]) for i in has_gold]) for k in ks}
        rows.append(dict(set=name, method=meth, **{f"recall@{k}": round(v, 3) for k, v in rec.items()}))
print(pd.DataFrame(rows).astype(object).fillna("").to_string(index=False))
print("(recall@30 on the Hybrid row is the ceiling: the reranker can't find what search didn't fetch)")

## 6. Verdict model on the top 5
Same model, same input as step 3. We now keep **all 5 passages' probabilities** (not just the max), because the aggregator uses them.

In [ ]:
vtok = AutoTokenizer.from_pretrained(MODEL_DIR)
vmodel = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEV).eval()
if DEV == "cuda": vmodel = vmodel.half()

@torch.no_grad()
def verdict_probs(pairs, bs=32, max_len=384):
    out = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i + bs]
        enc = vtok([c for c, _ in b], [d for _, d in b], truncation="only_second", max_length=max_len,
                   padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(vmodel(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def pair_probs(claims, rankings, topk=TOPK):
    # returns [n_claims, topk, 3]: P(SUP, REF, NEI) for each of the top-k passages
    pairs = [(c["claim"], corpus.text.iat[di]) for c, r in zip(claims, rankings) for di in r[:topk]]
    return verdict_probs(pairs).reshape(len(claims), topk, 3)

P_HYB = {n: pair_probs(c, RANKS[n]["Hybrid"]) for n, c in EVAL.items()}      # step 3 setup
P_RR  = {n: pair_probs(c, RERANKED[n]) for n, c in EVAL.items()}             # reranked setup
Y = {n: np.array([c["label"] for c in cl]) for n, cl in EVAL.items()}
print({n: P_RR[n].shape for n in EVAL})

## 7. Two ways to combine the 5 passages
**Max rule (step 3):** `s` = best SUPPORTS probability, `r` = best REFUTES probability. If `max(s, r) < τ` → NEI, else the larger wins.
τ is now chosen on the **two validation sets only** (average macro-F1), never on test.

**Learned aggregator:** logistic regression on 14 features per claim:
max and mean P(SUP), P(REF); the top-ranked passage's three probabilities; mean P(NEI); a reranker-weighted average of all three; the top reranker score, the gap between the top two scores, and the mean score. The reranker scores let it learn "the best passage isn't actually relevant, so say NEI".

In [ ]:
def decide(s, r, tau):
    return np.where(np.maximum(s, r) < tau, 2, np.where(s >= r, 0, 1))
def max_rule(P): return P[:, :, 0].max(1), P[:, :, 1].max(1)
F1 = lambda y, p: f1_score(y, p, average="macro")

VAL = ["SciFact val", "HealthVer val"]; TEST = "HealthVer test"
GRID = np.round(np.arange(0.3, 0.951, 0.05), 2)
def pick_tau(Pd):
    sc = [np.mean([F1(Y[n], decide(*max_rule(Pd[n]), t)) for n in VAL]) for t in GRID]
    return GRID[int(np.argmax(sc))], max(sc)
tau_hyb, _ = pick_tau(P_HYB); tau_rr, _ = pick_tau(P_RR)
print(f"tau chosen on validation only -> hybrid: {tau_hyb}, reranked: {tau_rr}")

def make_feats(P, S):
    S = S[:, :P.shape[1]]
    w = S / (S.sum(1, keepdims=True) + 1e-9)
    wp = (w[:, :, None] * P).sum(1)
    return np.column_stack([P[:, :, 0].max(1), P[:, :, 1].max(1), P[:, :, 0].mean(1), P[:, :, 1].mean(1),
                            P[:, 0, 0], P[:, 0, 1], P[:, 0, 2], P[:, :, 2].mean(1),
                            wp, S[:, 0], S[:, 0] - S[:, 1], S.mean(1)])

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict

X = {n: make_feats(P_RR[n], RSCORE[n]) for n in EVAL}
Xv = np.vstack([X[n] for n in VAL]); yv = np.concatenate([Y[n] for n in VAL])
strat = np.concatenate([Y[n] + 3 * i for i, n in enumerate(VAL)])          # stratify by label x set
mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000))
cv_pred = cross_val_predict(mk(), Xv, yv, cv=StratifiedKFold(5, shuffle=True, random_state=0).split(Xv, strat))
agg = mk().fit(Xv, yv)
offs = np.cumsum([0] + [len(Y[n]) for n in VAL])

rows = []
def add(label, f):
    rows.append({"method": label, **{n: round(f(n), 3) for n in EVAL}})
add("A. step 3: hybrid top-5, max rule", lambda n: F1(Y[n], decide(*max_rule(P_HYB[n]), tau_hyb)))
add("B. rerank top-5, max rule",         lambda n: F1(Y[n], decide(*max_rule(P_RR[n]), tau_rr)))
add("C. rerank top-5, learned aggregator", lambda n: F1(Y[n], cv_pred[offs[VAL.index(n)]:offs[VAL.index(n)+1]]) if n in VAL
                                                      else F1(Y[n], agg.predict(X[n])))
add("(ceiling) gold abstract given, step 2", lambda n: np.nan)
res = pd.DataFrame(rows); res.loc[3, "HealthVer test"] = 0.705
print("End-to-end macro-F1. Val columns for C are 5-fold cross-validated; HealthVer test is scored once.")
print(res.to_string(index=False))

**Reading the table.** A → B is the effect of the reranker. B → C is the effect of the learned aggregator. The last row is the upper bound (verdict model handed the right abstract).

In [ ]:
pred = agg.predict(X[TEST]); y = Y[TEST]
print("HealthVer test, learned aggregator (rows = true, cols = pred):")
print(pd.crosstab(pd.Series([LABELS[i] for i in y], name="true"), pd.Series([LABELS[i] for i in pred], name="pred")))
pa = decide(*max_rule(P_HYB[TEST]), tau_hyb)
nei = y == 2
print(f"\ntrue-NEI claims called SUPPORTS: step 3 = {int((pa[nei]==0).sum())}/{int(nei.sum())}  ->  now = {int((pred[nei]==0).sum())}/{int(nei.sum())}")

proba = agg.predict_proba(X[TEST]); conf = proba.max(1)
print("\nSelective prediction on HealthVer test (answer only when confident):")
rows = [dict(min_confidence=t, answered=f"{(conf >= t).mean():.0%}", accuracy_on_answered=round((pred[conf >= t] == y[conf >= t]).mean(), 3))
        for t in (0.0, 0.5, 0.6, 0.7, 0.8)]
print(pd.DataFrame(rows).to_string(index=False))

## 7b. Abstain unless the evidence is clearly relevant
In the demo, "if you don't drink water you will die" got a confident *Contradicted* from an unrelated alcohol paper. A **relevance gate** fixes the worst of that: if the best abstract's reranker score is below the gate, the answer is *not enough evidence*.

The table shows what each gate costs. *Wrong verdicts* = claims where the system says SUPPORTS or REFUTES and is wrong (the errors that matter most for a fact-checker). The gate is chosen on the **validation** sets only (cross-validated predictions): the **highest gate whose validation macro-F1 stays within 0.03 of the best**. Test is shown for reference, not used to choose.

In [ ]:
top = {n: RSCORE[n][:, 0] for n in EVAL}
base_pred = {n: (cv_pred[offs[VAL.index(n)]:offs[VAL.index(n) + 1]] if n in VAL else agg.predict(X[n])) for n in EVAL}
gated = lambda n, g: np.where(top[n] < g, 2, base_pred[n])
GATES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
def row(n, g):
    p = gated(n, g); y = Y[n]; said = p != 2
    return dict(f1=F1(y, p), verdicts=said.mean(), wrong=(said & (p != y)).mean())
rows = []
for g in GATES:
    v = [row(n, g) for n in VAL]; t = row(TEST, g)
    rows.append(dict(gate=g, val_F1=round(np.mean([r["f1"] for r in v]), 3), val_verdicts=f"{np.mean([r['verdicts'] for r in v]):.0%}",
                     val_wrong=f"{np.mean([r['wrong'] for r in v]):.0%}", test_F1=round(t["f1"], 3),
                     test_verdicts=f"{t['verdicts']:.0%}", test_wrong=f"{t['wrong']:.0%}"))
tab = pd.DataFrame(rows); print(tab.to_string(index=False))
best = tab.val_F1.max(); g_star = float(max(g for g, f in zip(tab.gate, tab.val_F1) if f >= best - 0.03))
print("\nchosen gate (validation rule):", g_star)

## 8. Try it on a free-text claim
This is exactly what the demo will call. Edit the claims and rerun.

In [ ]:
def check_claim(text, show=3):
    c = [dict(claim=text)]
    cand = fuse(rank_bm25([text]), rank_dense([text]))
    rk, S = rerank(c, cand)
    P = pair_probs(c, rk, TOPK)
    p = agg.predict_proba(make_feats(P, S))[0]
    order = list(agg.classes_)
    probs = {LABELS[k]: float(p[order.index(k)]) if k in order else 0.0 for k in range(3)}
    verdict = max(probs, key=probs.get)
    evid = [dict(uid=uid_of[di], title=corpus.title.iat[di], rerank=float(S[0, j]),
                 passage_verdict=LABELS[int(P[0, j].argmax())], passage_probs=P[0, j].round(3).tolist())
            for j, di in enumerate(rk[0][:show])]
    if evid[0]["rerank"] < g_star: verdict = "NEI (gated: no relevant abstract)"
    return dict(claim=text, verdict=verdict, confidence=round(probs[max(probs, key=probs.get)], 3), probs=probs, evidence=evid)

for t in ["Vitamin D supplementation reduces the risk of respiratory infections.",
          "Masks do not reduce the spread of COVID-19.",
          "Drinking lemon water cures cancer.", "If you don't drink water, you will die.",
          "If you do not eat food, you will die."]:
    r = check_claim(t)
    print(f"\n{r['claim']}\n  -> {r['verdict']} ({r['confidence']})  {r['probs']}")
    for e in r["evidence"]: print(f"     [{e['passage_verdict']:8s}] {e['rerank']:.2f}  {e['title'][:90]}")

## 9. Save everything the demo needs

In [ ]:
corpus.to_parquet(f"{OUT}/corpus.parquet", index=False)
np.save(emb_path, D)
if os.path.exists(f"{OUT}/aggregator.joblib") and not os.path.exists(f"{OUT}/aggregator_prev.joblib"):
    os.rename(f"{OUT}/aggregator.joblib", f"{OUT}/aggregator_prev.joblib")      # keep the previous (v2-based) aggregator
joblib.dump(agg, f"{OUT}/aggregator.joblib")
json.dump(dict(emb_model=EMB_MODEL, query_prefix=QUERY_PREFIX, topk=TOPK, ncand=NCAND, rerank_model=RERANK_MODEL,
               n_docs=len(corpus), tau_max_rule=float(tau_rr), verdict_model=VERDICT_DIR, relevance_gate=g_star,
               feature_order="see make_feats in step3b notebook"),
          open(f"{OUT}/retrieval_config.json", "w"))
print(sorted(os.listdir(OUT)))

## What to check
1. **Recall table**: does Hybrid+rerank beat Hybrid at @5 on HealthVer? It should move from about 0.71 toward 0.80+. If not, the ceiling (recall@30) is the limit and more candidates are the next lever.
2. **A → B → C**: each step should help or tie. If C is below B on HealthVer test, the aggregator overfit the small validation sets, so keep B.
3. **True-NEI called SUPPORTS**: should fall from 22/48.
4. **Selective prediction**: accuracy on answered claims should rise as the confidence threshold rises. That is the abstention story for the pitch.
5. **The three free-text claims**: sanity check only. Don't quote them as results.

In [ ]:
# Free the GPU before part 3 (the demo loads its own copies from Drive)
import gc
for n in ["vmodel", "vtok", "rr", "st", "model", "trainer"]:
    globals().pop(n, None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
assert os.path.exists(f"{DRIVE}/retrieval/aggregator.joblib"), "Part 2 did not save; fix it before running part 3"
print("Part 2 saved to Drive. Starting the demo.")

# Part 3 of 3 — Step 6: Veritas demo (Gradio)
Paste a transcript (or a YouTube link) → the app finds the checkable health claims → searches the evidence pool → returns a verdict, a confidence level and the supporting abstracts for each claim.

**Pipeline loaded from your Drive**
1. `claim_detector_v2` finds checkable claims (threshold from its config, adjustable in the UI)
2. hybrid search (BM25 + `bge-small`) → top 30 abstracts → `bge-reranker-base` → top 5
3. the verdict model named in the retrieval config (v3) judges each (claim, abstract) pair
4. `aggregator.joblib` combines them into one verdict + confidence
5. relevance gate (value from step 3b): if even the best abstract scores below it, the answer is "not enough evidence". Verdicts resting on loosely related abstracts (relevance under 0.6) carry a warning

**Use a fresh runtime:** Runtime → Disconnect and delete runtime, then **T4 GPU**, then Run all. Approve the Drive pop-up. The last cell prints a public link (it stays alive while this tab is open).

In [ ]:
import os, re, json, html as H, collections, joblib, numpy as np, pandas as pd, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification
DRIVE = DRIVE_ROOT   # set by the location cell near the top
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LABELS = ["SUPPORTS", "REFUTES", "NEI"]
MAX_CLAIMS = 30

cfg = json.load(open(f"{DRIVE}/retrieval/retrieval_config.json"))
VERDICT_DIR = cfg.get("verdict_model", "verdict_model_v2")      # the model the aggregator was trained on
GATE = float(cfg.get("relevance_gate", 0.15))                  # below this the answer is "not enough evidence" (chosen in step 3b)
WEAK = 0.6                                                     # below this, a verdict is shown with a "weak evidence" warning
EMB_MODEL, QUERY_PREFIX, TOPK, NCAND, RERANK_MODEL = cfg["emb_model"], cfg["query_prefix"], cfg["topk"], cfg["ncand"], cfg["rerank_model"]
corpus = pd.read_parquet(f"{DRIVE}/retrieval/corpus.parquet"); D = np.load(f"{DRIVE}/retrieval/corpus_emb.npy")
agg = joblib.load(f"{DRIVE}/retrieval/aggregator.joblib")
assert len(corpus) == len(D), "corpus and embeddings are out of sync"

half = lambda m: m.half() if DEV == "cuda" else m
vtok = AutoTokenizer.from_pretrained(f"{DRIVE}/{VERDICT_DIR}")
vmodel = half(AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/{VERDICT_DIR}").to(DEV).eval())
ctok = AutoTokenizer.from_pretrained(f"{DRIVE}/claim_detector_v2")
cmodel = half(AutoModelForSequenceClassification.from_pretrained(f"{DRIVE}/claim_detector_v2").to(DEV).eval())
CLAIM_THR = json.load(open(f"{DRIVE}/claim_detector_v2/extract_config.json"))["threshold"]
st = SentenceTransformer(EMB_MODEL, device=DEV); st.max_seq_length = 256
rr = CrossEncoder(RERANK_MODEL, max_length=384, device=DEV)
if DEV == "cuda": rr.model.half()

tok_re = re.compile(r"[a-z0-9]+")
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were will with".split())
toks = lambda t: [w for w in tok_re.findall(t.lower()) if w not in STOP]
bm25 = BM25Okapi([toks(t) for t in corpus.search_text])
print("loaded | device:", DEV, "| pool:", len(corpus), "| claim threshold:", CLAIM_THR, "| verdict model:", VERDICT_DIR, "| gate:", GATE)

## 1. The pipeline functions
Same code as steps 3b and 4b, batched so a whole transcript is processed together.

In [ ]:
def rank_bm25_q(qs, n=100): return [np.argsort(-bm25.get_scores(toks(q)))[:n] for q in qs]
def rank_dense_q(qs, n=100):
    Q = st.encode([QUERY_PREFIX + q for q in qs], batch_size=64, normalize_embeddings=True, convert_to_numpy=True)
    return [np.argsort(-s)[:n] for s in Q @ D.T]
def fuse(r1, r2, n=100, k=60):
    out = []
    for a, b in zip(r1, r2):
        sc = collections.defaultdict(float)
        for r in (a, b):
            for rank, i in enumerate(r): sc[i] += 1.0 / (k + rank + 1)
        out.append(np.array(sorted(sc, key=sc.get, reverse=True)[:n]))
    return out

def rerank(texts, rankings, ncand=NCAND, bs=64):
    pairs = [(t, corpus.search_text.iat[di]) for t, r in zip(texts, rankings) for di in r[:ncand]]
    sc = np.asarray(rr.predict(pairs, batch_size=bs, show_progress_bar=False), dtype="float64").reshape(len(texts), -1)
    if sc.min() < 0 or sc.max() > 1: sc = 1 / (1 + np.exp(-sc))
    nr, ns = [], []
    for r, s in zip(rankings, sc):
        o = np.argsort(-s); nr.append(r[:ncand][o]); ns.append(s[o])
    return nr, np.array(ns)

@torch.no_grad()
def verdict_probs(pairs, bs=32, max_len=384):
    out = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i + bs]
        enc = vtok([c for c, _ in b], [d for _, d in b], truncation="only_second", max_length=max_len, padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(vmodel(**enc).logits.float(), -1).cpu().numpy())
    return np.concatenate(out)

def make_feats(P, S):
    S = S[:, :P.shape[1]]; w = S / (S.sum(1, keepdims=True) + 1e-9); wp = (w[:, :, None] * P).sum(1)
    return np.column_stack([P[:, :, 0].max(1), P[:, :, 1].max(1), P[:, :, 0].mean(1), P[:, :, 1].mean(1),
                            P[:, 0, 0], P[:, 0, 1], P[:, 0, 2], P[:, :, 2].mean(1), wp, S[:, 0], S[:, 0] - S[:, 1], S.mean(1)])

def check_claims(texts, show=3):
    # batch: claims -> verdict, confidence, evidence
    rk, S = rerank(texts, fuse(rank_bm25_q(texts), rank_dense_q(texts)))
    pairs = [(t, corpus.text.iat[di]) for t, r in zip(texts, rk) for di in r[:TOPK]]
    P = verdict_probs(pairs).reshape(len(texts), TOPK, 3)
    A = agg.predict_proba(make_feats(P, S)); order = list(agg.classes_); out = []
    for i, t in enumerate(texts):
        probs = {LABELS[k]: float(A[i][order.index(k)]) if k in order else 0.0 for k in range(3)}
        gated = bool(S[i, 0] < GATE)
        verdict = "NEI" if gated else max(probs, key=probs.get)
        ev = [dict(title=corpus.title.iat[di] or "(untitled)", rerank=float(S[i, j]), passage=LABELS[int(P[i, j].argmax())],
                   p=float(P[i, j].max()), excerpt=corpus.text.iat[di][:420]) for j, di in enumerate(rk[i][:show])]
        out.append(dict(claim=t, verdict=verdict, confidence=None if gated else probs[verdict], gated=gated, weak=bool((not gated) and verdict != "NEI" and S[i, 0] < WEAK), probs=probs, evidence=ev))
    return out

@torch.no_grad()
def claim_probs(sents, bs=64):
    out = []
    for i in range(0, len(sents), bs):
        b = ctok(sents[i:i + bs], truncation=True, max_length=128, padding=True, return_tensors="pt").to(DEV)
        out.append(torch.softmax(cmodel(**b).logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.array([])

def split_transcript(text, max_words=40, window=25):
    text = re.sub(r"\[?\(?\b\d{1,2}:\d{2}(?::\d{2})?\b\]?\)?", " ", text)
    text = re.sub(r"\s+", " ", text).strip(); out = []
    for s in re.split(r"(?<=[.!?])\s+", text):
        w = s.split()
        if len(w) <= max_words:
            if len(w) >= 4: out.append(s)
        else:
            out += [" ".join(w[i:i + window]) for i in range(0, len(w), window) if len(w[i:i + window]) >= 4]
    return out

## 2. Turning results into a page
Each claim gets a card: verdict, a confidence level, and the top three abstracts with each abstract's own verdict. The transcript view highlights the claims in place.

In [ ]:
V = {"SUPPORTS": ("Supported by evidence", "#10b981", "#ecfdf5"), "REFUTES": ("Contradicted by evidence", "#ef4444", "#fef2f2"),
     "NEI": ("Not enough evidence", "#6b7280", "#f3f4f6")}
conf_label = lambda c: (lambda c: "high" if c >= 0.70 else "medium" if c >= 0.50 else "low")(round(c, 2))   # label matches the rounded percentage shown
INK = "color:#111827 !important;"
WRAP = lambda h: "<div style='background:#ffffff;padding:12px;border-radius:8px;color:#111827 !important'>" + h + "</div>"   # readable in dark mode

def card(r, i):
    name, border, bg = V[r["verdict"]]
    if r["gated"]:   conf = "no relevant abstract found in the evidence pool"
    elif r["weak"]:  conf = f"confidence low ({r['confidence']:.0%}) · only loosely related abstracts were found, so treat this with caution"
    else:            conf = f"confidence {conf_label(r['confidence'])} ({r['confidence']:.0%})"
    ev = "".join(
        f"<details style='margin:4px 0'><summary style='cursor:pointer;{INK}font-size:13px'>"
        f"<b style='{INK}'>{H.escape(e['title'][:110])}</b> &nbsp;<span style='color:#4b5563 !important'>reads as {e['passage'].lower()} ({e['p']:.0%}) · relevance {e['rerank']:.2f}</span></summary>"
        f"<div style='font-size:12px;color:#374151 !important;padding:4px 0 4px 12px'>{H.escape(e['excerpt'])}…</div></details>" for e in r["evidence"])
    return (f"<div style='border-left:6px solid {border};background:{bg};padding:10px 14px;margin:10px 0;border-radius:6px;{INK}'>"
            f"<div style='font-size:12px;color:#4b5563 !important'>CLAIM {i}</div>"
            f"<div style='font-size:15px;margin:2px 0 6px 0;{INK}'>{H.escape(r['claim'])}</div>"
            f"<div style='font-size:14px;{INK}'><b style='color:{border} !important'>{name}</b> &nbsp;·&nbsp; {conf}</div>"
            f"<div style='margin-top:6px;{INK}'>{ev}</div></div>")

def analyze(transcript, thr=CLAIM_THR):
    sents = split_transcript(transcript or "")
    if not sents: return WRAP("<i>Paste a transcript first.</i>"), ""
    p = claim_probs(sents); flagged = [i for i, x in enumerate(p) if x >= thr]
    note = ""
    if len(flagged) > MAX_CLAIMS:
        flagged = sorted(sorted(flagged, key=lambda i: -p[i])[:MAX_CLAIMS]); note = f" (showing the {MAX_CLAIMS} most claim-like)"
    res = check_claims([sents[i] for i in flagged]) if flagged else []
    by = {i: r for i, r in zip(flagged, res)}
    cnt = collections.Counter(r["verdict"] for r in res)
    summary = (f"<div style='{INK}font-size:14px;margin-bottom:6px'><b>{len(sents)}</b> sentences · <b>{len(flagged)}</b> checkable claims{note} · "
               f"<span style='color:#10b981 !important'><b>{cnt['SUPPORTS']}</b> supported</span> · <span style='color:#ef4444 !important'><b>{cnt['REFUTES']}</b> contradicted</span> · "
               f"<span style='color:#6b7280 !important'><b>{cnt['NEI']}</b> not enough evidence</span></div>")
    cards = "".join(card(r, n + 1) for n, r in enumerate(res)) or "<i>No checkable claims found. Try lowering the claim threshold.</i>"
    spans = []
    for i, s in enumerate(sents):
        if i in by:
            _, border, bg = V[by[i]["verdict"]]
            spans.append(f"<span style='background:{bg};border-bottom:3px solid {border};padding:1px 3px;{INK}'>{H.escape(s)}</span>")
        else:
            spans.append(f"<span style='color:#6b7280 !important'>{H.escape(s)}</span>")
    view = "<div style='line-height:1.9;font-size:14px'>" + " ".join(spans) + "</div>"
    return WRAP(summary + cards), WRAP(view)

## 3. YouTube captions
Pulls the caption track for a video. YouTube often blocks requests from cloud notebooks (Colab included). If this fails, paste the transcript instead (the **⋯ → Show transcript** panel on YouTube).

In [ ]:
def fetch_captions(url):
    m = re.search(r"(?:v=|youtu\.be/|shorts/|embed/)([A-Za-z0-9_-]{11})", url or "")
    if not m: return "", "Could not find a video id in that link."
    vid = m.group(1)
    try:
        from youtube_transcript_api import YouTubeTranscriptApi
        try:    segs = [s.text for s in YouTubeTranscriptApi().fetch(vid)]               # v1.x
        except AttributeError: segs = [s["text"] for s in YouTubeTranscriptApi.get_transcript(vid)]   # older versions
        return " ".join(segs), f"Fetched {len(segs)} caption lines."
    except Exception as e:
        return "", f"Could not fetch captions ({type(e).__name__}). Paste the transcript instead."


## 4. Test without the UI
Runs the pipeline on two invented transcripts. Check the verdicts look sensible before launching.

In [ ]:
DEMO1 = '''Hey everyone, welcome back to the channel. Today I want to talk about sleep and supplements. So first, I take magnesium glycinate every night and honestly it cures my insomnia. Studies show that vitamin D supplementation reduces the risk of respiratory infections. I personally think cold plunges are amazing. Masks do not reduce the spread of COVID-19, that's just a fact. Intermittent fasting is the best diet in my opinion. Drinking lemon water every morning cures cancer. Make sure you like and subscribe. The Eiffel Tower is 330 meters tall. Statins cause more harm than good for most people. Anyway, let me know what you think in the comments.'''
DEMO2 = '''Okay so, um, last month my doctor told me my cholesterol was high. A lot of people say eggs raise your cholesterol but the research really doesn't back that up. I swapped sugar for honey and my energy is way better. Drinking eight glasses of water a day is necessary for good health. Seed oils are poison, they cause inflammation throughout the body. Honestly this smoothie recipe is so good. Sunscreen causes more cancer than it prevents. If you walk ten thousand steps a day you will live longer. Alright, that's it for today, thanks for watching.'''
for name, T in [("DEMO1", DEMO1), ("DEMO2", DEMO2)]:
    sents = split_transcript(T); p = claim_probs(sents); fl = [i for i, x in enumerate(p) if x >= CLAIM_THR]
    print(f"\n{name}: {len(sents)} sentences, {len(fl)} claims")
    for i, r in zip(fl, check_claims([sents[i] for i in fl])):
        conf = "no relevant abstract" if r["gated"] else f"{r['confidence']:.2f}" + (" (weak)" if r["weak"] else "")
        print(f"  {r['verdict']:8s} {conf:>20s} | {r['claim'][:80]}")

## 5. Launch
Prints a public link you can open on any device or give to judges. It works only while this notebook keeps running.

In [ ]:
import gradio as gr

LIMITS = '''**How it works.** A claim detector picks checkable sentences. Each claim is searched against a pool of **5,505 research abstracts** (SciFact + HealthVer), re-ranked, and judged by a fine-tuned DeBERTa verdict model.

**Honest limits**
- **Coverage:** the evidence pool is small. Claims on topics outside it return *not enough evidence*, which means "not found here", not "false".
- **Accuracy:** on held-out HealthVer claims the system scores about 0.6 macro-F1 (0.705 if handed the right abstract). Expect mistakes. Confidence labels are rough guides.
- **Claim detection** was trained on political debates, health claims and a small amount of synthetic text. It over-flags personal anecdotes.
- Research abstracts are not medical advice. Check with a clinician.'''

with gr.Blocks(title="Veritas: health claim checker") as demo:
    gr.Markdown("# Veritas — health claim checker\nPaste a video transcript. Veritas finds the health claims and checks them against research abstracts.")
    with gr.Row():
        with gr.Column(scale=1):
            tx = gr.Textbox(label="Transcript", lines=12, placeholder="Paste a transcript here…")
            with gr.Row():
                url = gr.Textbox(label="…or a YouTube link", scale=3); fetch = gr.Button("Fetch captions", scale=1)
            status = gr.Markdown("")
            thr = gr.Slider(0.3, 0.99, value=CLAIM_THR, step=0.01, label="Claim threshold (lower = flag more sentences)")
            go = gr.Button("Check claims", variant="primary")
            gr.Examples(examples=[[DEMO1], [DEMO2]], inputs=[tx], label="Invented example transcripts")
        with gr.Column(scale=1):
            out = gr.HTML(label="Results")
            with gr.Accordion("Transcript view", open=False): view = gr.HTML()
    with gr.Accordion("How it works and honest limits", open=False): gr.Markdown(LIMITS)
    fetch.click(fetch_captions, [url], [tx, status])
    go.click(analyze, [tx, thr], [out, view])

demo.launch(share=True, debug=False)